## Prepare input dataset for RLS

In [2]:
import pandas as pd
import geopandas as gpd
import datetime
from math import floor

In [4]:
# Load files

surveys = pd.read_csv("/data/data/RLS/AODN 2024-09-17/IMOS_-_National_Reef_Monitoring_Network_Sub-Facility_-_Survey_metadata.csv", header=70, index_col = 'survey_id')
fish = pd.read_csv("/data/data/RLS/AODN 2024-09-17/IMOS_-_National_Reef_Monitoring_Network_Sub-Facility_-_Global_reef_fish_abundance_and_biomass.csv", header=70)
fish_aus = fish[fish['country'] == 'Australia']

surveys_aus = surveys[surveys['country'] == 'Australia']

/tmp/ipykernel_13112/463855929.py:4: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  fish = pd.read_csv("/data/data/RLS/AODN 2024-09-17/IMOS_-_National_Reef_Monitoring_Network_Sub-Facility_-_Global_reef_fish_abundance_and_biomass.csv", header=70)


In [17]:
# Fix date and time

mean_hour_flt = surveys_aus['hour'].dropna().apply(lambda x : int(x[0:2]) + int(x[3:5]) / 60).mean()
mean_hour = datetime.time(hour = floor(mean_hour_flt), minute=int(60*(mean_hour_flt-floor(mean_hour_flt))))
surveys_aus['hour'] = surveys_aus['hour'].fillna(str(mean_hour))
surveys_aus['eventDate'] = pd.to_datetime(surveys_aus['survey_date'] + ' ' + surveys_aus['hour'])

/tmp/ipykernel_6380/2450207718.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_6380/2450207718.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
# Pivot table to have one row per survey.

piv = pd.pivot_table(fish_aus, index = 'survey_id', columns = 'species_name', values='biomass', aggfunc = 'sum', fill_value = 0)
piv = piv.loc[:, (piv != 0).any(axis=0)]

data = pd.merge(surveys_aus, piv, how = 'left', left_on='survey_id', right_index = True, validate='1:1')

# Drop NAs
first_species_index = list(data.columns).index('eventDate') + 1
species_columns = data.columns[first_species_index:]

data.dropna(axis=0, how='any', subset = species_columns, inplace = True)

data.to_csv('/data/data/RLS/AODN 2024-09-17/database.csv')

In [9]:
### Database Cyril

df = pd.read_csv('/data/data/RLS/Cyril/rls_biomass.csv', index_col = 'survey_id')

mean_hour_flt = surveys['hour'].dropna().apply(lambda x : int(x[0:2]) + int(x[3:5]) / 60).mean()
mean_hour = datetime.time(hour = floor(mean_hour_flt), minute=int(60*(mean_hour_flt-floor(mean_hour_flt))))
surveys['hour'] = surveys['hour'].fillna(str(mean_hour))
surveys['eventDate'] = pd.to_datetime(surveys['survey_date'] + ' ' + surveys['hour'])

df = df.join(surveys[['eventDate', 'location', 'depth']], validate='1:1')

df.to_csv('/data/data/RLS/AODN 2024-09-17/database-cyril.csv')

In [13]:
# Create enrichment file

from geoenrich.dataloader import *
from geoenrich.enrichment import create_enrichment_file

geodf = import_occurrences_csv(path = '/data/data/RLS/AODN 2024-09-17/database-cyril.csv',
                               id_col = 'survey_id', date_col = 'eventDate', lat_col = 'latitude',
                               lon_col = 'longitude', depth_col = 'depth')

create_enrichment_file(geodf, 'rls_cyril')

4684 occurrences were loaded.
File saved at /data/data/geoenrich/biodiv/rls_cyril.csv


## Random split

In [14]:
import numpy as np
import pandas as pd 

def mapping(x):
    if x >= 0.8:
        return('test')
    elif x >= 0.6:
        return('val')
    else:
        return('train')

In [15]:
# Split by location

ds = pd.read_csv('/data/data/RLS/AODN 2024-09-17/database-cyril.csv', index_col = 'survey_id')

locations = ds[['location']].drop_duplicates()
np.random.seed(seed = 248)
locations['rand'] = np.random.random(size = len(locations))
locations['subset'] = locations['rand'].apply(mapping)
locations.drop(columns='rand', inplace=True)
ds['survey_id'] = ds.index
ds_split = ds.merge(locations, how = 'left', left_on='location', right_on = 'location', validate='m:1')
ds_split.set_index('survey_id').to_csv('/data/data/RLS/AODN 2024-09-17/database-cyril_split.csv')

In [17]:
## NSpecies = 558

Index(['latitude', 'longitude', 'site_code', 'Abudefduf bengalensis',
       'Abudefduf luridus', 'Abudefduf saxatilis', 'Abudefduf sexfasciatus',
       'Abudefduf troschelii', 'Abudefduf vaigiensis', 'Abudefduf whitleyi',
       ...
       'Thalassoma quinquevittatum', 'Upeneus tragula', 'Variola louti',
       'Zanclus cornutus', 'Zebrasoma scopas', 'Zebrasoma velifer',
       'eventDate', 'location', 'depth', 'survey_id'],
      dtype='object', length=565)

## Abundance bins

In [13]:
import numpy as np
import pandas as pd 

In [14]:
ds = pd.read_csv('/data/data/RLS/AODN 2024-09-17/database_split.csv', index_col = 'survey_id')

first_species_index = list(ds.columns).index('eventDate') + 1
species = ds.iloc[:, first_species_index:-1]

/tmp/ipykernel_55510/2924693781.py:1: DtypeWarning: Columns (22,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  ds = pd.read_csv('/data/data/RLS/AODN 2024-09-17/database_split.csv', index_col = 'survey_id')


In [15]:
num_bins = 25

all_data = species.to_numpy().flatten()
nonzero = all_data[all_data != 0]
bins = [0] + list(np.quantile(nonzero, np.linspace(0, 1, num_bins)))
bins[-1] += 1

In [18]:
for i in species.columns:
    ds[i] = pd.cut(ds[i], bins, include_lowest=True, labels= False, right = False)


In [19]:
ds.to_csv(f"/data/data/RLS/AODN 2024-09-17/database_split_binned-{num_bins}.csv")

## Inspect dataset

In [ ]:
import plotly.express as px

fig = px.scatter_geo(sites_aus, lat = sites_aus.geometry.y, lon = sites_aus.geometry.x,
                     hover_name = 'site_name', size = 'count')

fig.update_layout(width = 900)

fig.show()

In [30]:
ds = pd.read_csv('/data/data/RLS/AODN 2024-09-17/database.csv', index_col = 'survey_id')

# species
first_species_index = list(ds.columns).index('eventDate') + 1
species_number = len(ds.columns[first_species_index:])
species = ds.iloc[:, first_species_index:]
#((species > 0).sum(axis = 0) / len(ds)).hist(bins = 20)
print(f"Dataset consists of {len(species)} surveys of {species_number} unique species.")

/tmp/ipykernel_93821/1403555018.py:1: DtypeWarning: Columns (22,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  ds = pd.read_csv('/data/data/RLS/AODN 2024-09-17/database.csv', index_col = 'survey_id')


Dataset consists of 27437 surveys of 1796 unique species.


In [23]:
import numpy as np

species = ds.iloc[:, first_species_index:]
np.isnan(species.values).sum(axis=1)
nan_rows = ds.iloc[np.nonzero(np.isnan(species.values).sum(axis=1))[0]]
np.isnan(nan_rows.iloc[:, first_species_index:].values).sum()
np.product(nan_rows.iloc[:, first_species_index:].values.shape)

628600

## Merge population tiffs

In [ ]:
import xarray as xr
import glob

path = "/data/data/netcdf/gpw_v4_population_density/gpw_v4_population_density_adjusted_to_2015_unwpp_country_totals_rev11_{}_30_sec.nc"

list_da = []

for year in range(2000, 2021,5):
    da = xr.open_dataset(filename_or_obj=path.format(year))
    dt = pd.to_datetime(f"{year}-01-01")
    da = da.assign_coords(time = dt)
    da = da.expand_dims(dim="time")
    da.to_netcdf(path.format(year))

#stack dataarrays in list
#ds = xr.combine_by_coords(list_da)

ds#.to_netcdf(f"/data/data/netcdf/pop_density.nc")

## Get satellite data

In [33]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

from geoenrich.enrichment import load_enrichment_file, add_bounds

from PIL import Image
import pystac_client
import planetary_computer

import rasterio
from rasterio import windows, features, warp
from shapely import Polygon

from tqdm.auto import tqdm
tqdm.pandas()

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

geo_buffer = 5
sat_im_path = Path('/data/data/malpolon/inputs/landsat')
time_of_interest = "2020-01-01/2021-12-31"

df, _ = load_enrichment_file('rls_aus')
geodf = add_bounds(df[['geometry']], geo_buffer, time_buff=None)

27787 occurrences were loaded from enrichment file


In [47]:
def area_of_interest(row):
    aoi = {
            "type": "Polygon",
            "coordinates": [
                [
                    [row['minx'], row['miny']],
                    [row['minx'], row['maxy']],
                    [row['maxx'], row['maxy']],
                    [row['maxx'], row['miny']],
                    [row['minx'], row['miny']],
                ]
            ],
        }
    return(aoi)


def find_best_picture(row, collection = 'sentinel-2-l2a', asset='visual'):

    aoi_bounds = features.bounds(area_of_interest(row))

    query={"eo:cloud_cover": {"lt": 10}}

    if collection == 'landsat-c2-l2':
        query['platform'] = {"in": ["landsat-8", "landsat-9"]}

    search = catalog.search(
        collections=[collection],
        bbox=aoi_bounds,
        datetime=time_of_interest,
        query=query
    )  

    items = search.item_collection()

    if(len(items)):

        gdf = gpd.GeoDataFrame.from_features(items)
        gdf['id'] = [item.id for item in items]
        gdf['asset_href'] = [item.assets[asset].href for item in items]

        aoi = Polygon(area_of_interest(row)["coordinates"][0])

        selected = gdf[gdf.contains(aoi)]

        if len(selected):
            least_cloudy_item = selected.sort_values(by = 'eo:cloud_cover', axis=0).iloc[0]
            return(least_cloudy_item['asset_href'])

    
    return(None)



def download_picture(row):

    if row['asset_href']:

        with rasterio.open(row['asset_href']) as ds:
            aoi_bounds = features.bounds(area_of_interest(row))
            warped_aoi_bounds = warp.transform_bounds("epsg:4326", ds.crs, *aoi_bounds)
            aoi_window = windows.from_bounds(transform=ds.transform, *warped_aoi_bounds)
            band_data = ds.read(window=aoi_window)

            img = Image.fromarray(np.transpose(band_data, axes=[1, 2, 0]))
            img.save(sat_im_path / f"{row.name}.jpg")


def download_picture_landsat(row):

    if row['asset_href_blue']:

        bands = []
        for color in ['red', 'green', 'blue']:

            with rasterio.open(row['asset_href_' + color]) as ds:
                aoi_bounds = features.bounds(area_of_interest(row))
                warped_aoi_bounds = warp.transform_bounds("epsg:4326", ds.crs, *aoi_bounds)
                aoi_window = windows.from_bounds(transform=ds.transform, *warped_aoi_bounds)
                band_data = ds.read(window=aoi_window)

                bands.append(band_data[0])
        
        rgb = (0.0000275*np.stack(bands, axis = -1) - 0.2).clip(min=0, max=1)
        img = Image.fromarray(np.uint8(255*rgb))
        img.save(sat_im_path / f"{row.name}.jpg")


In [50]:

geodf2 = geodf3.iloc[0:1000]

geodf2['asset_href_blue'] = geodf2.progress_apply(find_best_picture, axis=1, args = ('landsat-c2-l2', 'blue'))
geodf2['asset_href_red'] = geodf2.progress_apply(find_best_picture, axis=1, args = ('landsat-c2-l2', 'red'))
geodf2['asset_href_green'] = geodf2.progress_apply(find_best_picture, axis=1, args = ('landsat-c2-l2', 'green'))
geodf2.progress_apply(download_picture_landsat, axis = 1)


  0%|          | 0/247 [00:00<?, ?it/s]

  0%|          | 0/247 [00:00<?, ?it/s]

  0%|          | 0/247 [00:00<?, ?it/s]

  0%|          | 0/247 [00:00<?, ?it/s]

id
923401442    None
923401457    None
923401429    None
923401472    None
923401439    None
             ... 
912340527    None
912340526    None
912340525    None
912340524    None
912344618    None
Length: 247, dtype: object

In [38]:
# Look for missing images

sat_path = Path('/data/data/malpolon/inputs/sat')
l = []

for id in geodf.index:
    if not(Path(sat_path / f"{id}.jpg").exists()):
        l.append(id)

geodf3 = geodf.loc[l]


## DHW Time series

In [5]:
from pathlib import Path
import xarray as xr
from datetime import datetime, timedelta
import numpy as np
from tqdm.auto import tqdm

root = Path('/mnt/marbec-data/FishNutBiogeo/noaa_5km_daily_dhw')
duration = 3652
lat = 0
lon = 0

In [6]:
end_date = '2020-01-01'
end_date_object = datetime.strptime(end_date, '%Y-%m-%d').date()
datelist = [end_date_object - timedelta(days=x) for x in range(duration)]
dayslist = [d.strftime("%Y%m%d") for d in datelist]
filelist = [root / f"year{m[:4]}"  / f"ct5km_dhw_v3.1_{m}.nc" for m in dayslist]

In [ ]:
d = {}

for day in tqdm(dayslist):
    file = root / f"year{day[:4]}"  / f"ct5km_dhw_v3.1_{day}.nc"
    with xr.open_mfdataset(file) as ds:
        d[day] = ds.sel(lat = lat, lon = lon, method = 'nearest')


  0%|          | 0/3652 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
from PIL import Image
from pathlib import Path
from scipy.ndimage import zoom
import numpy as np
from tqdm.auto import tqdm

directory = Path('/marbec-data/RLS-Australia/malpolon/inputs/australia/dhw-csv')
dhwmax = 33.72

for filepath in tqdm(directory.glob('*.csv')):

    data = pd.read_csv(filepath, index_col = 0)

    # # save max
    # if data['0'].max() > dhwmax:
    #     dhwmax = data['0'].max()

    values = data.iloc[:, 0].fillna(0)
    padded = np.pad(values, (0, 3652 - values.shape[0]), mode='constant').tolist()
    matrix = [padded[i*365:(i+1)*365] for i in range(10)]

    # Interpolate the matrix to 224x224
    zoom_factors = (224 / 10, 224 / 365)
    interpolated_matrix = zoom(matrix, zoom_factors)

    normalized_matrix = interpolated_matrix  / dhwmax
    np.save(filepath.parent.parent / 'dhw' / f"{filepath.stem}.npy")

print(dhwmax)

0it [00:00, ?it/s]

KeyboardInterrupt: 

In [13]:
data = pd.read_csv(filepath, index_col = 0, dtype = float)
data.iloc[:,0].tolist()

[nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan

## Inspect image sizes

In [27]:
file_list = list(sat_im_path.glob('*.jpg'))

d = {'w':[], 'h':[]}

for f in file_list:
    with Image.open(f) as im:
        width, height = im.size
        d['w'].append(width)
        d['h'].append(height)

df = pd.DataFrame(d)

In [28]:
df.describe()

,w,h
count,27432.000000,27432.000000
mean,1016.780548,1011.818205
std,9.332568,9.492992
min,1001.000000,995.000000
25%,1009.000000,1004.000000
50%,1016.000000,1011.000000
75%,1025.000000,1020.000000
max,1040.000000,1036.000000
